# E1 - Fashion-MNIST x MLP-5 (revision experiment)

Two-phase training (Phase 1 CE for 200 epochs, Phase 2 MSE for 600 epochs).
Same hyperparameters as the published MNIST baseline: MLP-5, width 512, ReLU,
lambda = 1e-4, Adam, cosine LR.

Three seeds. Per-epoch CSV (`fmnist_s{seed}.csv`) is saved after each seed so
nothing is lost if the runtime drops. Both NC1 < 0.01 and NC1 < 0.05 are
recorded.

Outputs:
- `fmnist_s0.csv`, `fmnist_s1.csv`, `fmnist_s2.csv`
- `fmnist_summary.csv` (T_NC, fn at NC1 thresholds 0.01 and 0.05)

In [1]:

import torch, torchvision, time, os, sys
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Runs on both Kaggle and Colab. Output dir is auto-detected.
if os.path.isdir('/kaggle/working'):
    PLATFORM = 'kaggle'
    SAVE_DIR = '/kaggle/working/'
    DATA_DIR = '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    # Mount Google Drive so per-seed CSVs survive a Colab session disconnect.
    # If mounting fails (e.g. permission denied), fall back to /content/ and
    # warn the user that the run is no longer crash-safe.
    DRIVE_DIR = '/content/drive/MyDrive/NC_revision/'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = DRIVE_DIR
        print(f'Drive mounted - outputs will be saved to {SAVE_DIR}')
    except Exception as exc:
        print(f'WARNING: Drive mount failed ({exc}). Falling back to /content/.')
        print('         A mid-run disconnect WILL lose progress.')
        SAVE_DIR = '/content/'
    # Datasets stay on local Colab disk so they do not eat Drive quota.
    DATA_DIR = '/content/data/'
else:
    PLATFORM = 'local'
    SAVE_DIR = './'
    DATA_DIR = './data/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), \
    'No GPU - Runtime -> Change runtime type -> A100 (Colab) or enable GPU (Kaggle)'
print(f'Platform: {PLATFORM}')
print(f'SAVE_DIR: {SAVE_DIR}')
print(f'GPU:      {torch.cuda.get_device_name(0)}')
print(f'Torch:    {torch.__version__}')

Mounted at /content/drive
Drive mounted - outputs will be saved to /content/drive/MyDrive/NC_revision/
Platform: colab
SAVE_DIR: /content/drive/MyDrive/NC_revision/
GPU:      NVIDIA A100-SXM4-80GB
Torch:    2.10.0+cu128


In [2]:

# Fashion-MNIST: 28x28, grayscale, 10 classes (Xiao et al., 2017).
# Normalization values are the dataset mean/std on the training set.
transform = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])

trainset = torchvision.datasets.FashionMNIST(
    DATA_DIR, train=True,  download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST(
    DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'Fashion-MNIST: {len(trainset):,} train / {len(testset):,} test')

100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.95MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 62.4MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.9MB/s]


Fashion-MNIST: 60,000 train / 10,000 test


In [3]:

class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU,
                 num_classes=10, in_dim=784):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(in_dim, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

print('MLP defined.')

MLP defined.


In [4]:

@torch.no_grad()
def compute_nc(model, loader, K):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c])
               for c in range(K)) / len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask] - (-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1 - (Mn * Wn).sum(1).mean()).item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3,
            'feat_norm': H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

print('NC metrics + evaluate ready.')

NC metrics + evaluate ready.


In [5]:

def run_twophase(model, name, lr=1e-3, wd=1e-4,
                 phase1=200, phase2=600, nc_every=10, K=10):
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception:
        pass
    model = model.to(DEVICE)
    rows = []; terminal = False
    t_nc_strict = None; fn_at_strict = None
    t_nc_relaxed = None; fn_at_relaxed = None
    t0 = time.time()

    for phase, loss_fn, n_ep in [(1, 'ce', phase1), (2, 'mse', phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase == 2 else 0

        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_fn == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y, K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sch.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep}')
                if not terminal and ep_l % (nc_every * 5) == 0:
                    print(f'  [{name}] ep={ep} tr={tr:.4f} te={te:.4f} '
                          f'(pre-terminal) t={(time.time()-t0)/60:.1f}m')
                if terminal:
                    nc = compute_nc(model, train_loader, K)
                else:
                    nc = {'nc1': None, 'nc2': None, 'nc3': None, 'feat_norm': None}
                rows.append({'epoch': ep, 'phase': phase,
                             'train': tr, 'test': te, **nc})
                # T_NC is defined as the first Phase-2 epoch with NC1
                # below the threshold. Phase-1 NC1 dips (seen e.g. with
                # Tanh) are transient and do not reflect equilibrium
                # collapse; the published numbers use the same definition.
                if nc['nc1'] is not None and phase == 2:
                    if t_nc_relaxed is None and nc['nc1'] < 0.05:
                        t_nc_relaxed = ep; fn_at_relaxed = nc['feat_norm']
                        print(f'  [{name}] NC1<0.05 at ep {ep} '
                              f'fn={fn_at_relaxed:.4f}')
                    if t_nc_strict is None and nc['nc1'] < 0.01:
                        t_nc_strict = ep; fn_at_strict = nc['feat_norm']
                        print(f'  [{name}] NC1<0.01 at ep {ep} '
                              f'fn={fn_at_strict:.4f}')

    elapsed = (time.time() - t0) / 60
    print(f'  [{name}] Done in {elapsed:.1f} min')
    return (pd.DataFrame(rows),
            t_nc_strict, fn_at_strict,
            t_nc_relaxed, fn_at_relaxed)

print('run_twophase ready.')

run_twophase ready.


In [6]:

results = []
for seed in range(3):
    print(f'\n=== Fashion-MNIST  seed={seed} ===')
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    model = MLP(depth=5, width=512, act_cls=nn.ReLU,
                num_classes=10, in_dim=784)
    df, ts, fs, tr_, fr = run_twophase(model, f'fmnist-s{seed}',
                                       lr=1e-3, wd=1e-4,
                                       phase1=200, phase2=600)
    df.to_csv(f'{SAVE_DIR}fmnist_s{seed}.csv', index=False)
    results.append({'seed': seed,
                    'T_NC_strict': ts, 'fn_strict': fs,
                    'T_NC_relaxed': tr_, 'fn_relaxed': fr,
                    'test_acc_final': df.test.iloc[-1]})
    pd.DataFrame(results).to_csv(f'{SAVE_DIR}fmnist_summary.csv', index=False)
    print(f'  saved fmnist_s{seed}.csv and fmnist_summary.csv')

summary = pd.DataFrame(results)
print('\n=== Fashion-MNIST summary ===')
print(summary.to_string(index=False))

ok = summary.dropna(subset=['fn_relaxed'])
if len(ok) >= 2:
    fns = ok['fn_relaxed'].values
    print(f'\nfn at NC1<0.05 (N={len(ok)}): '
          f'mean={fns.mean():.4f} std={fns.std():.4f} '
          f'CV={100*fns.std()/fns.mean():.1f}%')


=== Fashion-MNIST  seed=0 ===
  [fmnist-s0] Terminal phase at epoch 50
  [fmnist-s0] NC1<0.05 at ep 240 fn=1.3965
  [fmnist-s0] Done in 53.3 min
  saved fmnist_s0.csv and fmnist_summary.csv

=== Fashion-MNIST  seed=1 ===
  [fmnist-s1] Terminal phase at epoch 50
  [fmnist-s1] NC1<0.05 at ep 230 fn=1.2798
  [fmnist-s1] Done in 53.8 min
  saved fmnist_s1.csv and fmnist_summary.csv

=== Fashion-MNIST  seed=2 ===
  [fmnist-s2] ep=50 tr=0.9849 te=0.8911 (pre-terminal) t=2.9m
  [fmnist-s2] Terminal phase at epoch 60
  [fmnist-s2] NC1<0.05 at ep 240 fn=1.4253
  [fmnist-s2] Done in 53.3 min
  saved fmnist_s2.csv and fmnist_summary.csv

=== Fashion-MNIST summary ===
 seed T_NC_strict fn_strict  T_NC_relaxed  fn_relaxed  test_acc_final
    0        None      None           240    1.396518          0.8916
    1        None      None           230    1.279837          0.8934
    2        None      None           240    1.425259          0.8910

fn at NC1<0.05 (N=3): mean=1.3672 std=0.0629 CV=4.6%


In [7]:

# Save / download outputs.
# Kaggle: files in /kaggle/working/ are already persisted; nothing to do here.
# Colab : trigger a browser download for each file.
out_files = [f'{SAVE_DIR}fmnist_summary.csv'] + \
            [f'{SAVE_DIR}fmnist_s{s}.csv' for s in range(3)]
if PLATFORM == 'colab' and SAVE_DIR.startswith('/content/drive'):
    # Files are already persisted in Drive. No browser download needed.
    print('Files saved to Drive:', SAVE_DIR)
    for fp in out_files:
        if os.path.exists(fp):
            print(' ', fp, f'({os.path.getsize(fp)} bytes)')
elif PLATFORM == 'colab':
    from google.colab import files
    for fp in out_files:
        if os.path.exists(fp):
            files.download(fp)
else:
    print('Files saved in', SAVE_DIR)
    for fp in out_files:
        if os.path.exists(fp):
            print(' ', fp, f'({os.path.getsize(fp)} bytes)')

Files saved to Drive: /content/drive/MyDrive/NC_revision/
  /content/drive/MyDrive/NC_revision/fmnist_summary.csv (167 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_s0.csv (8080 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_s1.csv (8251 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_s2.csv (8015 bytes)
